# 機械学習の入口と学び方

この notebook は、機械学習章に入る前に『この章では何をしているのか』を迷わず掴むための導入です。単回帰、重回帰、分類、時系列、特徴量設計、SQL は別々の話に見えますが、実際には『何を当てたいのかを決める → 使ってよい情報を集める → うまくいったか確かめる → 改善する』という同じ流れの上にあります。

## 最初に捨てたい誤解

初学者が最初に詰まりやすいのは、機械学習を『強いモデル名を覚える科目』として読んでしまうことです。けれど実際には、モデル選びの前に『何を予測したいのか』『予測する時点で何を知っていてよいのか』『当たったと判断する基準は何か』を決める必要があります。

この章ではモデルそのものも学びますが、それと同じくらい『学習のためのデータをどう作るか』と『比べ方をどう揃えるか』を重視します。ここが曖昧なままでは、どんな高性能モデルを使っても結果の意味がぶれてしまいます。

## 機械学習はまず『何を当てたいのか』から始まる

たとえば『明日の売上を予測したい』『このメールが迷惑メールか知りたい』『来月解約しそうな人を見つけたい』のように、最初に立つのは素朴な問いです。機械学習は、その問いに対して過去の例を使いながら、似た状況ならどうなりそうかを当てにいく方法です。

だから最初に覚えるべきなのは難しい式ではありません。『何を当てたいのか』『その答えを当てる前に見てよい情報は何か』を整理する姿勢です。ここが定まると、後から出てくる回帰や分類の式は『その問いに答えるための道具』として読めるようになります。

## この章を読む前提

- Python 章の内容: 配列、DataFrame、可視化、基本的な関数定義
- 平均・差・割合のような基本計算
- グラフを見て『右上がり』『ばらつきが大きい』と言える程度の読解

線形代数や確率論を先に深く知らなくても、この章は読めます。ただし、出てきた式をすぐ一般論として暗記するより、まず小さなデータで『この式は何を比べているのか』『この数字は何を表しているのか』を言葉に戻す姿勢が重要です。

## 章全体の地図

1. 単回帰: 1つの入力から予測する最小形をつかむ
2. 重回帰: 入力が複数になると、読み取りがどう難しくなるかを見る
3. scikit-learn と XGBoost: モデルを同じ条件で比べる流れを作る
4. 特徴量エンジニアリング: 入力の作り方で性能がどう変わるかを見る
5. 教師あり / 教師なし: 問題の種類が変わると、見るべき指標も変わることを押さえる
6. 時系列: 未来の情報を混ぜない入力設計を学ぶ
7. SQL: 学習用の表を作る前段の考え方を固める

順番には理由があります。最初に『予測と誤差』を掴み、そのあとで『入力が増えると何が難しくなるか』へ進み、後半で『現実のデータ作りと評価設計』へ広げます。最初から全部を道具として覚えるより、『どんな問題で必要になったのか』で読む方が混乱しにくくなります。

## 初学者が最初につまずきやすい場所

- 回帰式やモデル名が出た時点で、もう理解した気になってしまう
- その時点で本当に使ってよい情報かを考えず、何でも入力に入れてしまう
- 途中で何度も答え合わせをして、最終的な比較を壊してしまう
- 時系列で未来の情報をうっかり混ぜてしまう
- SQL で表を作るときに、学習時点と予測時点を混同してしまう

この章では、難しい数式より先に、こうした実務的な誤読を減らすことを狙います。

In [ ]:
workflow = [
    '何を当てたいか決める',
    '使ってよい情報を決める',
    '練習用と確認用にデータを分ける',
    'まずは簡単な基準を置く',
    '改善して比べる',
]
for i, step in enumerate(workflow, start=1):
    print(i, step)

出力された順番は地味ですが、この骨格が崩れると学習結果の意味がかなり怪しくなります。機械学習では、強いモデルを入れるより前に、『ちゃんと比べられる練習問題を作る』方が重要な場面が多くあります。

ここで後から出てくる用語も少しだけ触れておくと、`target` は『当てたい答え』、`train/dev/test` は『練習用・途中確認用・最終確認用に分けたデータ』くらいに読めれば十分です。最初は英単語を覚えるより、役割を掴むことを優先してください。

## この章で繰り返し使う視点

- 入力: 予測するその時点で、本当に使ってよい情報だけを使っているか
- 学習: 何を良しとしてモデルを調整しているか説明できるか
- 評価: どのデータで、どの物差しを使って比べているか明確か
- 改善: 何を変えたから結果が動いたのか比較できるか

この 4 点を毎 notebook で意識すると、題材が回帰から時系列や SQL に変わっても学び方がぶれません。

In [ ]:
records = [
    {'day': 1, 'sales': 12},
    {'day': 2, 'sales': 15},
    {'day': 3, 'sales': 14},
]
target_day = 3
usable = [r['sales'] for r in records if r['day'] < target_day]
leaky = [r['sales'] for r in records if r['day'] <= target_day]
print('usable_past =', usable)
print('leaky_example =', leaky)

この差は小さく見えても、学習では決定的です。予測したい日そのものの値を混ぜた瞬間、モデルは未来を盗み見たことになります。時系列や SQL の後半ノートで出てくる `shift` や `snapshot` は、この『未来を混ぜない』ための具体策です。ここでも、まず単語より先に『予測の時点で知りえないものを入れてはいけない』という原則を掴むのが重要です。

## この章を読み終えたときの目標

この章の到達点は、単に回帰式や API を再現できることではありません。『いま見ている問題では、何を当てたいのか』『何を入力にしてよいのか』『どのデータでどう比べているのか』を一貫して説明できることです。その土台があると、より複雑な深層学習の章に進んでも、式やモデル名の多さに飲まれにくくなります。